In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # authorize once

Mounted at /content/drive




*   Collect for 2025
*   Collect for the not enough keywords again.
*   Delete from year data the keywords that you recollected.
*   Write a pipeline that concantates and clears the videos



In [ ]:
path = '/content/drive/MyDrive/Keyword-analysis/topics'

In [ ]:
#API keys:
FIRST_API = ''
SECOND_API = ''
THIRD_API = ''
FOURTH_API = ''
FIFTH_API = ''
SIXTH_API = ''
SEVENTH_API = ''
EIGTH_API = ''
NINTH_API = ''
TENTH_API = ''


API_11 = ''
API_12 = ''
API_13 = ''
API_14 = ''

In [ ]:
df.groupby('keywords').size()

,0
keywords,
autophagy,2711
cancer,3945
cardiovascular disease,3937
cellular senescence & senolytics,1565
centenarians,3196
dementia and alzheimer,3175
diabetes & glycemic control,3119
epigenetic and methylation,1884
exercise & physical activity,3882


In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
import time
import json

API_KEYS = [FIRST_API, SECOND_API, THIRD_API, FOURTH_API, FIFTH_API, SIXTH_API, SEVENTH_API, EIGTH_API, NINTH_API, TENTH_API, API_11, API_12, API_13, API_14]  # strings!

# "centenarians", "exercise & physical activity", "gut microbiome", "metformin", "rapamycin and sirolimus", "cellular senescence & senolytics", "telomere biology", "epigenetic and methylation",
#                  "mitochondria & mitophagy", "sirtuins", "autophagy", "diabetes & glycemic control", "cardiovascular disease", "obesity",
#                  "cancer", "modafinil and nootropics", "dementia and alzheimer"

#"nootropics, nootropics_2", "cellular senescence", "epigenetics", "mitochondria", "rapamycin"
SEARCH_TERMS = ["epigenetics", "mitochondria", "rapamycin"]
GENERAL_TERM = " and longevity and healthy aging"


# --- helpers ---
def get_youtube(api_key):
    return build("youtube", "v3", developerKey=api_key)

def is_quota_error(e: HttpError) -> bool:
    try:
        payload = json.loads(e.content.decode("utf-8"))
        reasons = [err.get("reason", "") for err in payload.get("error", {}).get("errors", [])]
        message = payload.get("error", {}).get("message", "")
        return any(r in ("quotaExceeded", "dailyLimitExceeded") for r in reasons) or "quota" in message.lower()
    except Exception:
        return "quota" in str(e).lower()

def with_key_rotation(call_fn, *args, **kwargs):
    """
    Run a Google API call function and rotate API keys if quota is exceeded.
    `call_fn` should be a lambda that uses the global `youtube`.
    """
    global key_index, youtube
    while True:
        try:
            return call_fn(*args, **kwargs)
        except HttpError as e:
            if is_quota_error(e):
                key_index += 1
                if key_index >= len(API_KEYS):
                    raise RuntimeError("All API keys exhausted!") from e
                print(f"⚠️ Quota exceeded. Switching to next API key ({key_index+1}/{len(API_KEYS)})")
                youtube = get_youtube(API_KEYS[key_index])
                # retry immediately with new key
                continue
            else:
                # Re-raise non-quota errors to be handled by caller if needed
                raise

def get_all_videos(query, year, max_pages=5):
    """Fetch up to max_pages of search results (50 each)."""
    nx_year = year + 1
    all_results = []
    next_page_token = None
    for _ in range(max_pages):
        resp = with_key_rotation(
            lambda: youtube.search().list(
                q=query,
                part="snippet",
                publishedAfter=f"{year}-01-01T00:00:00Z",
                publishedBefore=f"{nx_year}-01-01T00:00:00Z",
                maxResults=50,
                type="video",
                relevanceLanguage="en",
                pageToken=next_page_token
            ).execute()
        )
        all_results.extend(resp.get("items", []))
        next_page_token = resp.get("nextPageToken")
        if not next_page_token:
            break
    return all_results

def chunked(iterable, size):
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) == size:
            yield buf
            buf = []
    if buf:
        yield buf

In [ ]:
# --- main ---
key_index = 0
youtube = get_youtube(API_KEYS[key_index])

for term in SEARCH_TERMS:  # 2010–2025 inclusive
    video_data = []  # reset per year

    for year in range(2010, 2026):
        # 1) search (break after success)
        while True:
            try:
                search_term = term + GENERAL_TERM
                print(f"Searching with the term '{search_term}' for the year {year}")
                search_items = get_all_videos(search_term, year=year, max_pages=5)  # up to 250 per keyword
                break
            except HttpError as e:
                # non-quota HTTP error in search → log and skip this term
                print("HTTP error during search:", e)
                search_items = []
                break

        if not search_items:
            continue

        # 2) collect IDs and batch-fetch video details
        ids = [it.get("id", {}).get("videoId") for it in search_items if it.get("id", {}).get("videoId")]
        details_by_id = {}

        for id_chunk in chunked(ids, 50):
            try:
                v_resp = with_key_rotation(
                    lambda: youtube.videos().list(
                        part="statistics,snippet",
                        id=",".join(id_chunk)
                    ).execute()
                )
            except HttpError as e:
                print("HTTP error during videos.list:", e)
                continue

            for item in v_resp.get("items", []):
                details_by_id[item["id"]] = item

            time.sleep(0.2)  # gentle pacing

        # 3) assemble rows + (optional) top comments with key rotation
        for it in search_items:
            vid = it.get("id", {}).get("videoId")
            if not vid:
                continue

            detail = details_by_id.get(vid)
            if not detail:
                # video may be deleted/private or filtered out
                continue

            snippet = detail.get("snippet", {}) or {}
            stats = detail.get("statistics", {}) or {}
            lang = snippet.get("defaultAudioLanguage") or snippet.get("defaultLanguage")

            # language filter: skip non-English if language is known
            if lang and not str(lang).lower().startswith("en"):
                continue

            # try to fetch a few top-level comments (often fails / disabled)
            comment_text = ""
            try:
                c_resp = with_key_rotation(
                    lambda: youtube.commentThreads().list(
                        part="snippet",
                        videoId=vid,
                        maxResults=5,
                        textFormat="plainText"
                    ).execute()
                )
                comments = [
                    c["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
                    for c in c_resp.get("items", [])
                ]
                comment_text = " || ".join(comments)
            except HttpError:
                pass

            video_data.append({
                "source": "YouTube",
                "video_id": vid,
                "video_url": f"https://www.youtube.com/watch?v={vid}",
                "title": snippet.get("title") or it["snippet"]["title"],
                "description": snippet.get("description") or it["snippet"].get("description", ""),
                "date": snippet.get("publishedAt") or it["snippet"].get("publishedAt"),
                "channel": snippet.get("channelTitle") or it["snippet"].get("channelTitle", ""),
                "comments": comment_text,
                "views": stats.get("viewCount"),
                "likes": stats.get("likeCount"),
                "comments_count": stats.get("commentCount"),
                "keywords": term,
                "country": lang if lang else "unknown",
            })
            time.sleep(0.1)  # gentle pacing

    # 4) save ONCE per year (after all keywords)
    df = pd.DataFrame(video_data)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["year"] = year  # force the search year
        df["date"] = df["date"].dt.date
        # # optional: drop dupes across overlapping keywords
        # df = df.drop_duplicates(subset="video_id", keep="first")


    df.to_csv(f"{path}/{term}.csv", index=False)
    print(f"Done! Collected {len(df)} unique videos for term {term}")
    # no need to clear; it will be reinitialized next loop

Searching with the term 'epigenetics and longevity and healthy aging' for the year 2010


Searching with the term 'epigenetics and longevity and healthy aging' for the year 2011


Searching with the term 'epigenetics and longevity and healthy aging' for the year 2012


Searching with the term 'epigenetics and longevity and healthy aging' for the year 2013


Searching with the term 'epigenetics and longevity and healthy aging' for the year 2014


...

Searching with the term 'rapamycin and longevity and healthy aging' for the year 2022


Searching with the term 'rapamycin and longevity and healthy aging' for the year 2023


Searching with the term 'rapamycin and longevity and healthy aging' for the year 2024


Searching with the term 'rapamycin and longevity and healthy aging' for the year 2025


Done! Collected 2014 unique videos for term rapamycin


# Data fix    


In [ ]:
#collecting year columns
import pandas as pd
dfs = []
for year in range (2010, 2026):
  df = pd.read_csv(f'{path}/{year}.csv')
  dfs.append(df)


In [ ]:
import pandas as pd
df = pd.concat(dfs, ignore_index=True)
df.shape

(43903, 14)

In [ ]:
#collecting keyword columns
terms = ["nootropics", "nootropics_2", "cellular senescence", "epigenetics", "mitochondria", "rapamycin"]
df_terms = []
for keyword in terms:
  df = pd.read_csv(f"{path}/{keyword}.csv")
  df = df.drop(columns=['year'])
  df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
  df["year"] = df["date"].dt.year       # extract year while it's datetime
  df["date"] = df["date"].dt.date
  df_terms.append(df)

In [ ]:
dfk = pd.concat(df_terms, ignore_index=True)
dfk.shape

(13963, 14)

In [ ]:
dfk.groupby('keywords').size()

,0
keywords,
cellular senescence,3158
epigenetics,3177
mitochondria,3638
nootropics,1976
rapamycin,2014


In [ ]:
df.groupby('keywords').size()

,0
keywords,
autophagy,2711
cancer,3945
cardiovascular disease,3937
cellular senescence & senolytics,1565
centenarians,3196
dementia and alzheimer,3175
diabetes & glycemic control,3119
epigenetic and methylation,1884
exercise & physical activity,3882


In [ ]:
#from df: delete - "cellular senescence & senolytics", "epigenetic and methylation", "mitochondria & mitophagy"
delete_terms = ["cellular senescence & senolytics", "epigenetic and methylation", "mitochondria & mitophagy", "rapamycin and sirolimus"]
df = df[~df['keywords'].isin(delete_terms)]
df.groupby('keywords').size()

,0
keywords,
autophagy,2711
cancer,3945
cardiovascular disease,3937
centenarians,3196
dementia and alzheimer,3175
diabetes & glycemic control,3119
exercise & physical activity,3882
gut microbiome,2463
metformin,2497


In [ ]:
#in dfk change following:
rename_map = {
    "cellular senescence": "cellular senescence & senolytics",
    "epigenetics": "epigenetic and methylation",
    "mitochondria": "mitochondria & mitophagy",
    "nootropics": "modafinil and nootropics",
    "rapamycin": "rapamycin and sirolimus"
}


dfk["keywords"] = dfk["keywords"].replace(rename_map)
dfk.groupby('keywords').size()

,0
keywords,
cellular senescence & senolytics,3158
epigenetic and methylation,3177
mitochondria & mitophagy,3638
modafinil and nootropics,1976
rapamycin and sirolimus,2014


In [ ]:
df_combined = pd.concat([df, dfk], ignore_index=True)
df_combined.groupby('keywords').size()

,0
keywords,
autophagy,2711
cancer,3945
cardiovascular disease,3937
cellular senescence & senolytics,3158
centenarians,3196
dementia and alzheimer,3175
diabetes & glycemic control,3119
epigenetic and methylation,3177
exercise & physical activity,3882


In [ ]:
df_combined.columns

Index(['source', 'video_id', 'video_url', 'title', 'description', 'date',
       'channel', 'comments', 'views', 'likes', 'comments_count', 'keywords',
       'country', 'year'],
      dtype='object')

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
df_combined.groupby(['year', 'keywords']).size()

year  keywords                        
2010  autophagy                            15
      cancer                              249
      cardiovascular disease              245
      cellular senescence & senolytics     41
      centenarians                        128
      dementia and alzheimer               47
      diabetes & glycemic control          29
      epigenetic and methylation           45
      exercise & physical activity        180
      gut microbiome                       10
      metformin                            15
      mitochondria & mitophagy            100
      modafinil and nootropics              3
      obesity                             180
      rapamycin and sirolimus               4
      sirtuins                             56
      telomere biology                     52
2011  autophagy                            54
      cancer                              249
      cardiovascular disease              249
      cellular senescence & senolytics     77
      centenarians                        132
      dementia and alzheimer               68
      diabetes & glycemic control         100
      epigenetic and methylation           86
      exercise & physical activity        249
      gut microbiome                       20
      metformin                            24
      mitochondria & mitophagy            163
      modafinil and nootropics             12
      obesity                             249
      rapamycin and sirolimus              12
      sirtuins                             92
      telomere biology                     93
2012  autophagy                            45
      cancer                              248
      cardiovascular disease              246
      cellular senescence & senolytics    143
      centenarians                        132
      dementia and alzheimer              108
      diabetes & glycemic control         136
      epigenetic and methylation          119
      exercise & physical activity        246
      gut microbiome                       18
      metformin                            34
      mitochondria & mitophagy            228
      modafinil and nootropics             15
      obesity                             248
      rapamycin and sirolimus              27
      sirtuins                            111
      telomere biology                    113
2013  autophagy                            58
      cancer                              248
      cardiovascular disease              248
      cellular senescence & senolytics    121
      centenarians                        178
      dementia and alzheimer              164
      diabetes & glycemic control         114
      epigenetic and methylation          136
      exercise & physical activity        250
      gut microbiome                       29
      metformin                            44
      mitochondria & mitophagy            209
      modafinil and nootropics             31
      obesity                             249
      rapamycin and sirolimus              32
      sirtuins                             88
      telomere biology                    128
2014  autophagy                            64
      cancer                              246
      cardiovascular disease              247
      cellular senescence & senolytics    181
      centenarians                        182
      dementia and alzheimer              174
      diabetes & glycemic control         138
      epigenetic and methylation          166
      exercise & physical activity        248
      gut microbiome                       62
      metformin                            51
      mitochondria & mitophagy            245
      modafinil and nootropics             38
      obesity                             248
      rapamycin and sirolimus              29
      sirtuins                             96
      telomere biology                    149
2015  autophagy                           117
     

In [ ]:
df_combined.to_csv(f'{path}/data.csv', index=False)

## Data preprocessing

In [ ]:
import pandas as pd
df_youtube = pd.read_csv("/content/drive/MyDrive/Keyword-analysis/clean_YouTube.csv")

In [ ]:
df_youtube.columns

Index(['source', 'video_id', 'video_url', 'title', 'description', 'date',
       'channel', 'comments', 'views', 'likes', 'comments_count', 'keywords',
       'country', 'year', 'likes_missing', 'topic_num', 'text', 'text_clean'],
      dtype='object')

In [ ]:
import pandas as pd
df = pd.read_csv(f'{path}/data.csv')
df.columns

Index(['source', 'video_id', 'video_url', 'title', 'description', 'date',
       'channel', 'comments', 'views', 'likes', 'comments_count', 'keywords',
       'country', 'year'],
      dtype='object')

In [ ]:
print(df.groupby('keywords').size())
print(df.groupby('year').size())

keywords
autophagy                           2711
cancer                              3945
cardiovascular disease              3937
cellular senescence & senolytics    3158
centenarians                        3196
dementia and alzheimer              3175
diabetes & glycemic control         3119
epigenetic and methylation          3177
exercise & physical activity        3882
gut microbiome                      2463
metformin                           2497
mitochondria & mitophagy            3638
modafinil and nootropics            2142
obesity                             3840
rapamycin and sirolimus             2014
sirtuins                            2741
telomere biology                    2843
dtype: int64
year
2010    1399
2011    1929
2012    2217
2013    2327
2014    2564
2015    2779
2016    3106
2017    3499
2018    3719
2019    4018
2020    4061
2021    4099
2022    4192
2023    4247
2024    4148
2025    4174
dtype: int64


In [ ]:
df.groupby(['year', 'keywords']).size()

year  keywords                        
2010  autophagy                            15
      cancer                              249
      cardiovascular disease              245
      cellular senescence & senolytics     41
      centenarians                        128
                                         ... 
2025  modafinil and nootropics            259
      obesity                             243
      rapamycin and sirolimus             244
      sirtuins                            243
      telomere biology                    242
Length: 272, dtype: int64

In [ ]:
df.shape

(52478, 14)

In [ ]:
df.isnull().sum()

,0
source,0
video_id,0
video_url,0
title,0
description,2521
date,0
channel,0
comments,24837
views,10
likes,1288


In [ ]:
df[df['comments_count'].isnull()].isnull().sum()

,0
source,0
video_id,0
video_url,0
title,0
description,143
date,0
channel,0
comments,2471
views,0
likes,623


In [ ]:
df['comments_count']=df['comments_count'].fillna(0)

In [ ]:
# check comments_count and comments correnspond
df[df['comments'].isnull()].groupby('comments_count').size()

,0
comments_count,
0.0,24814
1.0,21
2.0,2


In [ ]:
df[(df['comments'].isnull()) & (df['comments_count']==2)]

,source,video_id,video_url,title,description,date,channel,comments,views,likes,comments_count,keywords,country,year
4605,YouTube,KulAWj6z5rs,https://www.youtube.com/watch?v=KulAWj6z5rs,Coping With Stress: Dr. Martinez interview wit...,http://www.amazon.com/gp/product/B008VGVU8Q\nC...,2013-06-17,Dr. Mario Martinez,NaN,9147.0,68.0,2.0,centenarians,en,2013
4625,YouTube,KulAWj6z5rs,https://www.youtube.com/watch?v=KulAWj6z5rs,Coping With Stress: Dr. Martinez interview wit...,http://www.amazon.com/gp/product/B008VGVU8Q\nC...,2013-06-17,Dr. Mario Martinez,NaN,9147.0,68.0,2.0,centenarians,en,2013


In [ ]:
#pipeline

def duplicated_deal(df):
  print("Number of duplicates")
  print(df['video_id'].duplicated().sum())
  df = df.drop_duplicates(subset="video_id", keep="last")
  print("After cleaning")
  print(df['video_id'].duplicated().sum())
  return df



def null_values(df):
  df['description'] = df['description'].fillna('')
  df['comments']=df['comments'].fillna('')
  df = df[~df['views'].isnull()]
  df['likes_missing'] = df['likes'].isnull()
  df['likes'] = df['likes'].fillna(0)
  return df


In [ ]:
df = duplicated_deal(df)

Number of duplicates
20294
After cleaning
0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
source,0
video_id,0
video_url,0
title,0
description,1780
date,0
channel,0
comments,16522
views,7
likes,846


In [ ]:
df = null_values(df)

/tmp/ipython-input-2858789091.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['likes_missing'] = df['likes'].isnull()
/tmp/ipython-input-2858789091.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['likes'] = df['likes'].fillna(0)


In [ ]:
df.isnull().sum()

,0
source,0
video_id,0
video_url,0
title,0
description,0
date,0
channel,0
comments,0
views,0
likes,0


In [ ]:
df[df['likes_missing']==True].shape

(846, 15)

In [ ]:
pd.set_option('display.max_rows', None)
df.groupby(['year', 'keywords']).size()

year  keywords                        
2010  autophagy                             4
      cancer                              157
      cardiovascular disease              102
      cellular senescence & senolytics     24
      centenarians                         92
      dementia and alzheimer               42
      diabetes & glycemic control           8
      epigenetic and methylation           39
      exercise & physical activity        126
      gut microbiome                        2
      metformin                             3
      mitochondria & mitophagy             98
      modafinil and nootropics              3
      obesity                             110
      rapamycin and sirolimus               4
      sirtuins                             25
      telomere biology                     17
2011  autophagy                            15
      cancer                              154
      cardiovascular disease               95
      cellular senescence & senolytics     41
      centenarians                         80
      dementia and alzheimer               57
      diabetes & glycemic control          31
      epigenetic and methylation           59
      exercise & physical activity        146
      gut microbiome                        9
      metformin                             5
      mitochondria & mitophagy            156
      modafinil and nootropics              7
      obesity                             160
      rapamycin and sirolimus              12
      sirtuins                             43
      telomere biology                     28
2012  autophagy                            11
      cancer                              114
      cardiovascular disease               97
      cellular senescence & senolytics     63
      centenarians                         79
      dementia and alzheimer               79
      diabetes & glycemic control          58
      epigenetic and methylation           85
      exercise & physical activity        172
      gut microbiome                       12
      metformin                             9
      mitochondria & mitophagy            219
      modafinil and nootropics              4
      obesity                             141
      rapamycin and sirolimus              27
      sirtuins                             60
      telomere biology                     31
2013  autophagy                            19
      cancer                              140
      cardiovascular disease               89
      cellular senescence & senolytics     63
      centenarians                        104
      dementia and alzheimer              126
      diabetes & glycemic control          43
      epigenetic and methylation           95
      exercise & physical activity        171
      gut microbiome                       16
      metformin                            11
      mitochondria & mitophagy            195
      modafinil and nootropics             15
      obesity                             149
      rapamycin and sirolimus              32
      sirtuins                             44
      telomere biology                     33
2014  autophagy                            25
      cancer                              120
      cardiovascular disease               75
      cellular senescence & senolytics    103
      centenarians                        117
      dementia and alzheimer              130
      diabetes & glycemic control          66
      epigenetic and methylation          120
      exercise & physical activity        151
      gut microbiome                       24
      metformin                            14
      mitochondria & mitophagy            206
      modafinil and nootropics             25
      obesity                             155
      rapamycin and sirolimus              29
      sirtuins                             50
      telomere biology                     39
2015  autophagy                            43
     

In [ ]:
df.groupby('keywords').size()

,0
keywords,
autophagy,1265
cancer,2153
cardiovascular disease,1639
cellular senescence & senolytics,1802
centenarians,2057
dementia and alzheimer,2487
diabetes & glycemic control,1829
epigenetic and methylation,2277
exercise & physical activity,2645


In [ ]:
# your dict: id -> name
topic_names = {
    0: "centenarians",
    1: "cellular senescence & senolytics",
    2: "telomere biology",
    3: "rapamycin and sirolimus",
    4: "metformin",
    5: "dementia and alzheimer",
    6: "exercise & physical activity",
    7: "diabetes & glycemic control",
    8: "gut microbiome",
    9: "epigenetic and methylation",
    10: "mitochondria & mitophagy",
    11: "sirtuins",
    12: "autophagy",
    13: "cardiovascular disease",
    14: "obesity",
    15: "cancer",
    16: "modafinil and nootropics",
}

# invert: name -> id
name_to_id = {v: k for k, v in topic_names.items()}

# exact match mapping (strings in df['keywords'])
df["topic_num"] = df["keywords"].map(name_to_id)  # unmatched => NaN

In [ ]:
df.isnull().sum()

,0
source,0
video_id,0
video_url,0
title,0
description,0
date,0
channel,0
comments,0
views,0
likes,0


In [ ]:
df.head()

,source,video_id,video_url,title,description,date,channel,comments,views,likes,comments_count,keywords,country,year,likes_missing,topic_num
3,YouTube,_5eV1cXUKJI,https://www.youtube.com/watch?v=_5eV1cXUKJI,Is the Secret to Long Life Genetic?,Living a long life may have more to do with DN...,2010-07-02,ABC News,I agree,1792.0,19.0,1.0,centenarians,en,2010,False,0
4,YouTube,aBd7HFIVwyg,https://www.youtube.com/watch?v=aBd7HFIVwyg,Health Secrets of the Centenarians,Health secrets of those who are living or have...,2010-11-28,God Removing The Veil,,245.0,2.0,0.0,centenarians,en,2010,False,0
7,YouTube,I03tBKVrHdk,https://www.youtube.com/watch?v=I03tBKVrHdk,Georgia Centenarian Study (Pt. 4): Phase 1 - C...,"This is an excerpt on ""Phase 1 - A Cross-Secti...",2010-11-11,ugagerontology,,3803.0,8.0,0.0,centenarians,en,2010,False,0
11,YouTube,XWE4YCQPHm8,https://www.youtube.com/watch?v=XWE4YCQPHm8,Georgia Centenarian Study (Pt. 11): Health & N...,"This is an excerpt on ""health & nutrition"" fro...",2010-11-11,ugagerontology,I read a book about many studies on the placeb...,1701.0,14.0,2.0,centenarians,en,2010,False,0
12,YouTube,ACENBcyOAcU,https://www.youtube.com/watch?v=ACENBcyOAcU,Georgia Centenarian Study (Pt. 9): Engaged Lif...,"This is an excerpt on ""engaged lifestyle & qua...",2010-11-11,ugagerontology,,1203.0,4.0,0.0,centenarians,en,2010,False,0


In [ ]:
print(df.groupby('topic_num').size().sort_values())
print()
print(df.groupby('keywords').size().sort_values())

topic_num
4      804
2     1195
12    1265
11    1386
8     1450
13    1639
16    1747
1     1802
7     1829
3     1836
0     2057
15    2153
9     2277
14    2436
5     2487
6     2645
10    3169
dtype: int64

keywords
metformin                            804
telomere biology                    1195
autophagy                           1265
sirtuins                            1386
gut microbiome                      1450
cardiovascular disease              1639
modafinil and nootropics            1747
cellular senescence & senolytics    1802
diabetes & glycemic control         1829
rapamycin and sirolimus             1836
centenarians                        2057
cancer                              2153
epigenetic and methylation          2277
obesity                             2436
dementia and alzheimer              2487
exercise & physical activity        2645
mitochondria & mitophagy            3169
dtype: int64


In [ ]:
df.shape

(32177, 16)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.to_csv(f'{path}/final_data.csv', index=False)